# 1.0 Project Overview

### 1.1 Real Life Problem

Traffic accidents are a common and persistent public safety issue worldwide, occurring daily in both urban and rural settings. According to the World Health Organization, road traffic accidents result in over a million fatalities each year, with many people experiencing both health and financial losses. In major cities, these accidents can be contributed by either:

* Human related factors such as distracted driving, speeding or impaired driving.

* Vehicle related factors such as vehicle defect, age of the vehicle and vehicle model.

* Environmental related factors such as poor weather, low visibility and road surface issues such as poorly maintained roads.

Understanding these factors is essential to reducing traffic accidents and developing effective safety measures.


### 1.2 Invested stakeholders

Several groups might benefit significantly from the traffic accidents data and insights drawn from understanding the primary contributory causes. These might include:

1. Government and city authorities - who might use this to identify high risk areas and improve on road infrastructure while implementing policies aimed at reducing accidents.

2. Public safety organizations - such as the Vehicle Safety Board who might use the data to aid in designing regulation and safety campaigns and enforcement strategies based on observed trends.

3. General public and drivers - who might benefit from better awareness of risk factors.

4. Vehicle manufacturers - who might leverage on insights to improve car safety features such as weather detection systems.

# 2.0 Business Understanding

### 2.1 Business Problem
Large cities such as the City of Chicago experience frequent traffic accidents often. To address this challenge, there is need for a predictive model that can identify **primary contributory cause of car accidents** using key influencing factors including:

1. Human Factors such as sex and age.

2. Vehicle Factors such as age of the vehicle and year of make.

3. Environmental Factors such as weather patterns.

4. Temporal Factors such as Date and Time of the Accidents.

5. Road Design Characteristics such as road defects and road surface conditions.

Such a model would support data driven decision making aimed at reducing accident rates and improving overall road safety.

### 2.2 Business Questions
The project seeks to answer the following business questions:

1. What are the most common primary contributory causes of traffic accidents?

2. How do human characteristics contribute to accident occurrence?

3. How do vehicle characteristics contribute to accident occurrence?

4. How do environmental conditions such as weather influence accident occurrence?

5. Are there specific times when accidents are more likely to occur?

6. How does road design impact the likelihood of accidents?

7. Can we accurately predict the primary cause of an accident given the human, vehicle, environmental, temporal and road design characteristics?

### 2.3 Business Objectives
The main objectives of this project are to:

1. Develop a predictive model that can classify the primary contributory cause of traffic accidents.

2. Identify key factors (Human, Vehicle, Environmental, Temporal) that significantly influence accident occurrence.


# 3.0 Data Understanding

### 3.1 Data Description and Sources

* This project utilized 3 datasets that contain information on the accident occurence, characteristics of people involved as well as the vehicle characteristics. These are:

1. City of Chicago Accident Dataset - sourced from https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if.
This dataset has 1.04 million rows and 48 columns. Each row is a traffic crash with the row identifier as the CRASH_RECORD_ID.
2. Vehicle Accident Dataset sourced from https://data.cityofchicago.org/Transportation/Traffic-Crashes-Vehicles/68nd-jvt3.  It has 2.28 million rows and 29 columns. Each row is a person with the row identifier as the PERSON_ID. The dataset also has CRASH_RECORD_ID.
3. Driver/Passenger Accident Dataset sourced from https://data.cityofchicago.org/Transportation/Traffic-Crashes-People/u6pd-qa9d. It has 2.12 million rows with 71 columns. Each row is a vehicle with the row identifier as the CRASH_UNIT_ID. This dataset however also has a CRASH_RECORD_ID.

**Note**: Our goal is to merge the three datasets using the common CRASH_RECORD_ID, creating a unified dataset that combines all relevant features for our analysis.

From the merger, we aim to use the following columns from the following datasets:

1. City of Chicago Dataset: 

* Weather condition - Weather condition at time of crash.
* Lightning condition - Light condition at time of crash.
* Road defects - Road defects, as determined by reporting officer.
* Crash type - A general severity classification for the crash. Can be either Injury and/or Tow Due to Crash or No Injury / Drive Away.
* Primary Contributory Cause - The factor which was most significant in causing the crash.
* Crash hour - The hour of the day component of CRASH_DATE.
* Crash day - The day of the week component of CRASH_DATE.
* Crash month - The month component of CRASH_DATE.

2. Vehicle Dataset:

* Number of passengers - Number of passengers in the vehicle. The driver is not included.
* Make - The make (brand) of the vehicle.
* Vehicle Year - The model year of the vehicle.
* Vehicle Use - The normal use of the vehicle.
* Vehicle Defect - Defect the vehicle had.
* Exceeds speed limit - Whether vehicle exceeded speed limit.

3. Passenger Dataset:

* Person type - Type of roadway user involved in crash.
* Sex - Gender of person involved in crash.
* Age - Age of person involved in crash.
* Driver Vision - What, if any, objects obscured the driver’s vision at time of crash.
* Physical Condition - Driver’s apparent physical condition at time of crash.
* Driver Action - Driver action that contributed to the crash.
* Pedpedal Action - Action of pedestrian or cyclist at the time of crash.



### 3.2 Target Variable
* Our target variable is: Primary contributory cause which is a multiclass classification problem.
There are 40 values of the aforementioned target variable and we aim to predict the most common primary contributory causes based on our feature variables.

### 3.3 Feature Variables
* All the other aforementioned variables other than primary contributory cause will act as our feature variables. They will be specifically selected following a merger of the 3 Datasets.

### 3.4 Challenge with large samples of rows
Following the merger, we expect 2 challenges:

1. Since a single crash may involve more than just the driver, each CRASH_RECORD_ID could correspond to multiple people. To simplify the analysis, we will focus on drivers, examining each crash event from the perspective of the drivers involved.
2. A single crash can involve multiple vehicles, so if a crash involves two cars with one person in each, it would result in at least two rows—one for each individual involved in the same crash.
3. The dataset contains millions of rows, which may exceed available computing resources. To manage this, we will limit our analysis to approximately 500,000 randomly selected samples after data cleaning, which will also help reduce the dataset size naturally.

# 4.0 Data Preparation and Cleaning

### 4.1 Data Preparation

* In this section we will load the data, select the columns we want to work with and limit it only to drivers

In [20]:
# We will start by importing the necessary libraries for data cleaning and visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [21]:
# Loading the 3 Datasets and saving them under different variable names. Starting with accidents_data and checking our data.

accidents_log = pd.read_csv("accidents.csv", low_memory = False) # Low memory helps pandas to read the entire file at once when determining column types
accidents_log.head()

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,c4658ba1425251e03a299caf165125eba8af03229e2238...,NaN,03/24/2026 11:10:00 PM,20,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,CLEAR,DARKNESS,FIXED OBJECT,OTHER,...,NaN,NaN,NaN,NaN,23,3,3,41.876214,-87.749114,POINT (-87.749114396596 41.876214475743)
1,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,NaN,03/24/2026 10:45:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",PARKED MOTOR VEHICLE,ONE-WAY,...,0.0,0.0,2.0,0.0,22,3,3,41.774542,-87.604726,POINT (-87.604726471916 41.774541988748)
2,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,NaN,03/24/2026 10:25:00 PM,10,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",REAR TO SIDE,PARKING LOT,...,0.0,0.0,2.0,0.0,22,3,3,41.696314,-87.596632,POINT (-87.596632243933 41.696313683973)
3,4ca882872708f63d8e3f3d30f0b4cd885132273e9b7708...,NaN,03/24/2026 10:18:00 PM,40,NO CONTROLS,NO CONTROLS,CLEAR,DARKNESS,PARKED MOTOR VEHICLE,NOT DIVIDED,...,0.0,0.0,3.0,0.0,22,3,3,41.802117,-87.622206,POINT (-87.622206182567 41.802117124614)
4,f8c941528e1bdd2a112d6c25ea37e289e92e724498b198...,NaN,03/24/2026 10:09:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",ANGLE,FOUR WAY,...,1.0,0.0,1.0,0.0,22,3,3,41.765109,-87.644557,POINT (-87.644556951763 41.765109431911)


In [22]:
# Loading the passenger data and checking our data
passenger_log = pd.read_csv("passenger.csv", low_memory = False)
passenger_log.head()

,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O2271292,DRIVER,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,2165998.0,03/24/2026 10:45:00 PM,NaN,DAVENPORT,IA,52804,F,...,NaN,OTHER,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,P499550,PASSENGER,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,2165998.0,03/24/2026 10:45:00 PM,3.0,ROCK ISLND,IL,61201,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,O2271266,DRIVER,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,2165976.0,03/24/2026 10:25:00 PM,NaN,CHICAGO,IL,60617,M,...,NaN,IMPROPER BACKING,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
3,O2271267,DRIVER,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,2165981.0,03/24/2026 10:25:00 PM,NaN,CHICAGO,IL,60628,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O2271268,DRIVER,4ca882872708f63d8e3f3d30f0b4cd885132273e9b7708...,2165979.0,03/24/2026 10:18:00 PM,NaN,CHICAGO,IL,60827,F,...,NaN,UNKNOWN,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [23]:
# Loading the vehicle data and checking our data
vehicle_log = pd.read_csv("vehicles.csv", low_memory = False)
vehicle_log.head()

,CRASH_UNIT_ID,CRASH_RECORD_ID,CRASH_DATE,UNIT_NO,UNIT_TYPE,NUM_PASSENGERS,VEHICLE_ID,CMRC_VEH_I,MAKE,MODEL,...,TRAILER1_LENGTH,TRAILER2_LENGTH,TOTAL_VEHICLE_LENGTH,AXLE_CNT,VEHICLE_CONFIG,CARGO_BODY_TYPE,LOAD_TYPE,HAZMAT_OUT_OF_SERVICE_I,MCS_OUT_OF_SERVICE_I,HAZMAT_CLASS
0,2271287,c4658ba1425251e03a299caf165125eba8af03229e2238...,03/24/2026 11:10:00 PM,1,DRIVERLESS,NaN,2165992.0,NaN,FORD,FUSION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2271292,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,03/24/2026 10:45:00 PM,1,DRIVER,1.0,2165998.0,NaN,GENERAL MOTORS CORPORATION (GMC),ACADIA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2271293,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,03/24/2026 10:45:00 PM,2,PARKED,NaN,2166003.0,NaN,CHRYSLER,OTHER (EXPLAIN IN NARRATIVE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2271266,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,03/24/2026 10:25:00 PM,1,DRIVER,NaN,2165976.0,NaN,CHEVROLET,IMPALA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2271267,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,03/24/2026 10:25:00 PM,2,DRIVER,NaN,2165981.0,NaN,LEXUS,300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Checking the different columns in each dataset
accidents_log.columns, passenger_log.columns, vehicle_log.columns

(Index(['CRASH_RECORD_ID', 'CRASH_DATE_EST_I', 'CRASH_DATE',
        'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION',
        'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE',
        'TRAFFICWAY_TYPE', 'LANE_CNT', 'ALIGNMENT', 'ROADWAY_SURFACE_COND',
        'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE', 'INTERSECTION_RELATED_I',
        'NOT_RIGHT_OF_WAY_I', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
        'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
        'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE',
        'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'DOORING_I', 'WORK_ZONE_I',
        'WORK_ZONE_TYPE', 'WORKERS_PRESENT_I', 'NUM_UNITS',
        'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
        'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
        'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
        'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
     

In [25]:
# Select ONLY the specific columns we outlined in Section 3.1
crash_cols = ['CRASH_RECORD_ID', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'ROAD_DEFECT', 
              'CRASH_TYPE', 'PRIM_CONTRIBUTORY_CAUSE', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH']

vehicle_cols = ['CRASH_RECORD_ID', 'OCCUPANT_CNT', 'MAKE', 'VEHICLE_YEAR', 
                'VEHICLE_USE', 'VEHICLE_DEFECT', 'EXCEED_SPEED_LIMIT_I']

passenger_cols = ['CRASH_RECORD_ID', 'PERSON_TYPE', 'SEX', 'AGE', 'DRIVER_VISION', 
                  'PHYSICAL_CONDITION', 'DRIVER_ACTION', 'PEDPEDAL_ACTION']

# Subset the dataframes to drop the 100+ columns we don't need
accidents_sub = accidents_log[crash_cols]
vehicle_sub = vehicle_log[vehicle_cols]
passenger_sub = passenger_log[passenger_cols]

In [26]:
# Going forward, we want to focus our dataset purely on factors that affect drivers
# Of the 3 datasets, we can specify drivers only in the passenger_log data as this has the persons column where we can specify the person involved in the crash
# The other datasets do not have 'persons' column to specify the person involved in the crash.
# However, after specifying the drivers and merging the datasets, we expect to have one dataset where the data concerns only drivers

drivers_only = passenger_sub[passenger_sub['PERSON_TYPE'] == 'DRIVER'].copy()

In [27]:
# Drop duplicates just in case there are multiple drivers/vehicles recorded per crash
# drivers_unique = drivers_only.drop_duplicates(subset=['CRASH_RECORD_ID'])
# vehicles_unique = vehicle_sub.drop_duplicates(subset=['CRASH_RECORD_ID'])

In [28]:
# We now want to merge the 3 datasets; accidents, passenger, and vehicles csv datasets
# From the above, we notice that the 3 datasets share the 'CRASH_RECORD_ID' column. We will therefore use pandas.merge to merge them based on this.
# We will first merge accidents_sub with vehicles_unique

merged_accidents_vehicles = pd.merge(
    accidents_sub,
    vehicle_sub,
    how = 'left',
    on = 'CRASH_RECORD_ID'
)

# we will then merge the combined merged_accidents_vehicles with drivers_unique to have 1 full dataset called merged_df
merged_df = pd.merge(
    merged_accidents_vehicles,
   drivers_only,
    how = 'left',
    on = 'CRASH_RECORD_ID'
)

# Check the merged_df
merged_df.head()

,CRASH_RECORD_ID,WEATHER_CONDITION,LIGHTING_CONDITION,ROAD_DEFECT,CRASH_TYPE,PRIM_CONTRIBUTORY_CAUSE,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,OCCUPANT_CNT,...,VEHICLE_USE,VEHICLE_DEFECT,EXCEED_SPEED_LIMIT_I,PERSON_TYPE,SEX,AGE,DRIVER_VISION,PHYSICAL_CONDITION,DRIVER_ACTION,PEDPEDAL_ACTION
0,c4658ba1425251e03a299caf165125eba8af03229e2238...,CLEAR,DARKNESS,UNKNOWN,INJURY AND / OR TOW DUE TO CRASH,FAILING TO REDUCE SPEED TO AVOID CRASH,23,3,3,0.0,...,PERSONAL,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,CLEAR,"DARKNESS, LIGHTED ROAD",UNKNOWN,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,22,3,3,2.0,...,PERSONAL,UNKNOWN,NaN,DRIVER,F,32.0,NOT OBSCURED,NORMAL,OTHER,NaN
2,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,CLEAR,"DARKNESS, LIGHTED ROAD",UNKNOWN,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,22,3,3,0.0,...,PERSONAL,UNKNOWN,NaN,DRIVER,F,32.0,NOT OBSCURED,NORMAL,OTHER,NaN
3,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,CLEAR,"DARKNESS, LIGHTED ROAD",NO DEFECTS,NO INJURY / DRIVE AWAY,IMPROPER BACKING,22,3,3,1.0,...,PERSONAL,NONE,NaN,DRIVER,M,40.0,NOT OBSCURED,NORMAL,IMPROPER BACKING,NaN
4,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,CLEAR,"DARKNESS, LIGHTED ROAD",NO DEFECTS,NO INJURY / DRIVE AWAY,IMPROPER BACKING,22,3,3,1.0,...,PERSONAL,NONE,NaN,DRIVER,M,50.0,NOT OBSCURED,NORMAL,NONE,NaN


In [29]:
# Checking the length of the merged datasets
len(merged_df) # we are dealing with 2,634,133 million entries

2634133

In [30]:
# checking the number of columns
len(merged_df.columns) # we have a total of 22 columns

22

In [31]:
merged_df.head()

,CRASH_RECORD_ID,WEATHER_CONDITION,LIGHTING_CONDITION,ROAD_DEFECT,CRASH_TYPE,PRIM_CONTRIBUTORY_CAUSE,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,OCCUPANT_CNT,...,VEHICLE_USE,VEHICLE_DEFECT,EXCEED_SPEED_LIMIT_I,PERSON_TYPE,SEX,AGE,DRIVER_VISION,PHYSICAL_CONDITION,DRIVER_ACTION,PEDPEDAL_ACTION
0,c4658ba1425251e03a299caf165125eba8af03229e2238...,CLEAR,DARKNESS,UNKNOWN,INJURY AND / OR TOW DUE TO CRASH,FAILING TO REDUCE SPEED TO AVOID CRASH,23,3,3,0.0,...,PERSONAL,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,CLEAR,"DARKNESS, LIGHTED ROAD",UNKNOWN,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,22,3,3,2.0,...,PERSONAL,UNKNOWN,NaN,DRIVER,F,32.0,NOT OBSCURED,NORMAL,OTHER,NaN
2,169bb4297ce8f9de0b432baae8e5c61fccff5f877be0e0...,CLEAR,"DARKNESS, LIGHTED ROAD",UNKNOWN,NO INJURY / DRIVE AWAY,UNABLE TO DETERMINE,22,3,3,0.0,...,PERSONAL,UNKNOWN,NaN,DRIVER,F,32.0,NOT OBSCURED,NORMAL,OTHER,NaN
3,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,CLEAR,"DARKNESS, LIGHTED ROAD",NO DEFECTS,NO INJURY / DRIVE AWAY,IMPROPER BACKING,22,3,3,1.0,...,PERSONAL,NONE,NaN,DRIVER,M,40.0,NOT OBSCURED,NORMAL,IMPROPER BACKING,NaN
4,ebc0228a98d221f831a0f232dc0c310c2d15d47e1a976b...,CLEAR,"DARKNESS, LIGHTED ROAD",NO DEFECTS,NO INJURY / DRIVE AWAY,IMPROPER BACKING,22,3,3,1.0,...,PERSONAL,NONE,NaN,DRIVER,M,50.0,NOT OBSCURED,NORMAL,NONE,NaN


In [32]:
 #Save the clean dataset into our folder
merged_df.to_csv('chicago_crashes_cleaned.csv', index=False)

### 4.2 Data Cleaning

* In this section we will explore the datatypes, look for missing values and fill them.
* We will not drop duplicates because we are assuming that each CRASH_RECORD_ID is shared between 2 drivers; the driver who caused the accident and the one affected. We will therefore treat each driver row as an independent accident.

In [33]:
# Exploring the datatypes

merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2634133 entries, 0 to 2634132
Data columns (total 22 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   CRASH_RECORD_ID          str    
 1   WEATHER_CONDITION        str    
 2   LIGHTING_CONDITION       str    
 3   ROAD_DEFECT              str    
 4   CRASH_TYPE               str    
 5   PRIM_CONTRIBUTORY_CAUSE  str    
 6   CRASH_HOUR               int64  
 7   CRASH_DAY_OF_WEEK        int64  
 8   CRASH_MONTH              int64  
 9   OCCUPANT_CNT             float64
 10  MAKE                     str    
 11  VEHICLE_YEAR             float64
 12  VEHICLE_USE              str    
 13  VEHICLE_DEFECT           str    
 14  EXCEED_SPEED_LIMIT_I     str    
 15  PERSON_TYPE              str    
 16  SEX                      str    
 17  AGE                      float64
 18  DRIVER_VISION            str    
 19  PHYSICAL_CONDITION       str    
 20  DRIVER_ACTION            str    
 21  PEDPEDAL_ACTION    

Of the 21 variables, 6 are numerical: CRASH_HOUR, CRASH_DAY_OF_WEEK, CRASH_MONTH, OCCUPANT_COUNT, VEHICLE_YEAR, AGE.

The remaining variables are all of the string datatype.

In [34]:
# we will now check for missing data
merged_df.isna().sum()

CRASH_RECORD_ID                  0
WEATHER_CONDITION                0
LIGHTING_CONDITION               0
ROAD_DEFECT                      0
CRASH_TYPE                       0
PRIM_CONTRIBUTORY_CAUSE          0
CRASH_HOUR                       0
CRASH_DAY_OF_WEEK                0
CRASH_MONTH                      0
OCCUPANT_CNT                 51319
MAKE                         51324
VEHICLE_YEAR                431714
VEHICLE_USE                  51319
VEHICLE_DEFECT               51319
EXCEED_SPEED_LIMIT_I       2631727
PERSON_TYPE                1441609
SEX                        1441609
AGE                        1758329
DRIVER_VISION              1441609
PHYSICAL_CONDITION         1441609
DRIVER_ACTION              1441609
PEDPEDAL_ACTION            2634133
dtype: int64

The following variables had missing values: 

* OCCUPANT_CNT                 51319
* MAKE                         51324
* VEHICLE_YEAR                431714
* VEHICLE_USE                  51319
* VEHICLE_DEFECT               51319
* EXCEED_SPEED_LIMIT_I       2631727
* PERSON_TYPE                1441609
* SEX                        1441609
* AGE                        1758329
* DRIVER_VISION              1441609
* PHYSICAL_CONDITION         1441609
* DRIVER_ACTION              1441609
* PEDPEDAL_ACTION            2634133

In [35]:
# Given that we had 2,634,133 entries and the column with the highest missing values (PEDPEDAL_ACTION) had 2,634,133 missing entries, and EXCEED_SPEED_LIMIT_I had 2,631,727 missing entries, we will drop both columns.

merged_df.drop(columns=['EXCEED_SPEED_LIMIT_I','PEDPEDAL_ACTION'],inplace=True)

# Confirming that we have deopped the 2 columns
merged_df.columns   # we have dropped the 2 columns

Index(['CRASH_RECORD_ID', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
       'ROAD_DEFECT', 'CRASH_TYPE', 'PRIM_CONTRIBUTORY_CAUSE', 'CRASH_HOUR',
       'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'OCCUPANT_CNT', 'MAKE',
       'VEHICLE_YEAR', 'VEHICLE_USE', 'VEHICLE_DEFECT', 'PERSON_TYPE', 'SEX',
       'AGE', 'DRIVER_VISION', 'PHYSICAL_CONDITION', 'DRIVER_ACTION'],
      dtype='str')

In [36]:
# Of the remaining variables, the one with the highest missing values is age with 1,758,329 missing values out of 2,634,133 total entries.
# This constitiutes 66.7% of the total data. Filling in these values with either mean or median might lead to erroneous data. 
# We therefore opt to drop these missing values completely.

merged_df.dropna(inplace=True)

#confirming that we have dropped all missing values
merged_df.isna().sum()  #We have dropped all missing values

CRASH_RECORD_ID            0
WEATHER_CONDITION          0
LIGHTING_CONDITION         0
ROAD_DEFECT                0
CRASH_TYPE                 0
PRIM_CONTRIBUTORY_CAUSE    0
CRASH_HOUR                 0
CRASH_DAY_OF_WEEK          0
CRASH_MONTH                0
OCCUPANT_CNT               0
MAKE                       0
VEHICLE_YEAR               0
VEHICLE_USE                0
VEHICLE_DEFECT             0
PERSON_TYPE                0
SEX                        0
AGE                        0
DRIVER_VISION              0
PHYSICAL_CONDITION         0
DRIVER_ACTION              0
dtype: int64

In [37]:
# Confirming the new number of tatal entries
len(merged_df)      #We are now working with 801,601 total complete entries.

801601

In [38]:
 #Save the final cleaned dataset into our folder
merged_df.to_csv('chicago_crashes_final_dataset_cleaned.csv', index=False)